# How to Use Configuration Presets

This notebook demonstrates how to use the `MatcherPresets` system to quickly configure matchers for common scenarios.

Configuration presets provide well-tested starting points that balance speed, accuracy, and computational cost for different use cases.

**Topics covered:**
1. Available presets and their characteristics
2. Using presets with overrides
3. Comparing presets
4. Creating and registering custom presets
5. Saving and loading configurations

## Setup

In [ ]:
from geomfum.dataset import NotebooksDataset
from geomfum.dataset.torch import MeshDataset, PairsDataset
from geomfum.experiment import ExperimentSuite, MatcherPresets
from geomfum.matcher import FunctionalMapMatcher, MatcherConfig
from geomfum.shape import TriangleMesh

## Available Presets

Let's explore what presets are available and their characteristics.

In [ ]:
# List all available presets
presets = MatcherPresets.list_presets()
print("Available presets:")
for preset in presets:
    print(f"  - {preset}")

In [ ]:
# Examine each preset's configuration

for preset in presets:
    config_dict = MatcherPresets.describe(preset)
    print(f"\n{'=' * 60}")
    print(f"Preset: {preset}")
    print(f"{'=' * 60}")

    # Show key parameters
    for key in ["spectrum_size", "fmap_size", "sdp_weight", "lb_weight", "mult_weight"]:
        if key in config_dict:
            print(f"  {key:15s}: {config_dict[key]}")

    # Show descriptor info
    if "descriptors" in config_dict and config_dict["descriptors"]:
        print(f"  descriptors    : {len(config_dict['descriptors'])} descriptor(s)")

    # Show refiner info
    if "refiners" in config_dict and config_dict["refiners"]:
        print(f"  refiners       : {len(config_dict['refiners'])} refiner(s)")
    elif "refiners" in config_dict:
        print("  refiners       : None (no refinement)")

### Preset Characteristics

**minimal**: Bare minimum settings for very fast testing
- Small spectrum (50), tiny functional map (10)
- No operator commutativity constraints
- No refinement
- **Use when**: Quick sanity checks, debugging

**quick**: Fast matching with acceptable accuracy
- Medium spectrum (100), small functional map (20)
- Basic constraints (SDP, LB, mult)
- Minimal refinement (5 ICP iterations)
- **Use when**: Rapid prototyping, parameter exploration

**standard**: Balanced speed/accuracy (recommended default)
- Standard spectrum (200), medium functional map (30)
- Full constraints
- Standard refinement (ICP + ZoomOut)
- **Use when**: Most production use cases

**precise**: High accuracy for benchmarking
- Large spectrum (300), large functional map (50)
- Full constraints including orientation
- Requires landmarks
- Extended refinement
- **Use when**: Final benchmarks, publications

**no_refinement**: Standard without post-processing
- Same as standard but no refinement
- **Use when**: Comparing raw vs. refined results

## Using Presets

Let's see how to use presets in practice.

In [ ]:
# Load example shapes
dataset = NotebooksDataset()

mesh_a = TriangleMesh.from_file(dataset.get_filename("faust-00"))
mesh_b = TriangleMesh.from_file(dataset.get_filename("faust-04"))

print(f"Shape A: {mesh_a.n_vertices} vertices")
print(f"Shape B: {mesh_b.n_vertices} vertices")

In [ ]:
# Use a preset directly
config = MatcherPresets.get("standard")
matcher = FunctionalMapMatcher(config=config)

result = matcher(mesh_a, mesh_b)
print(f"P2P correspondence shape: {result.p2p21.shape}")
print(f"Functional map shape: {result.fmap12.shape}")

### Using Presets with Overrides

You can start from a preset and override specific parameters.

In [ ]:
# Start from 'standard' but increase descriptor preservation weight
config = MatcherPresets.get("standard", sdp_weight=2.0, lb_weight=1e-1)

print("Modified config:")
print(f"  sdp_weight: {config.sdp_weight}")
print(f"  lb_weight: {config.lb_weight}")
print(f"  spectrum_size: {config.spectrum_size} (from preset)")

## Comparing Presets

Let's compare different presets on a small dataset.

In [ ]:
# Load a small dataset for comparison
dataset_dir = "../../../datasets/faust/test_set"

mesh_dataset = MeshDataset(
    dataset_dir=dataset_dir,
    spectral=True,
    distances=True,
    correspondences=True,
    k=30,
)

pairs_dataset = PairsDataset(
    dataset=mesh_dataset,
    pair_mode="random",
    pairs_ratio=0.2,  # Small subset for demo
)

print(f"Testing on {len(pairs_dataset)} pairs")

In [ ]:
# Create matchers with different presets
methods = {
    "minimal": FunctionalMapMatcher(config=MatcherPresets.get("minimal")),
    "quick": FunctionalMapMatcher(config=MatcherPresets.get("quick")),
    "standard": FunctionalMapMatcher(config=MatcherPresets.get("standard")),
    "no_refinement": FunctionalMapMatcher(config=MatcherPresets.get("no_refinement")),
}

# Run comparison
suite = ExperimentSuite(methods, pairs_dataset)
results = suite.run()

In [ ]:
# Print comparison table
suite.print_comparison(metrics=["geodesic_error", "coverage", "dirichlet_energy"])

## Creating Custom Presets

You can register your own presets for project-specific configurations.

In [ ]:
from geomfum.descriptor.pipeline import ArangeSubsampler
from geomfum.descriptor.spectral import HeatKernelSignature, WaveKernelSignature
from geomfum.refine import IcpRefiner

# Create a custom configuration
custom_config = MatcherConfig(
    spectrum_size=150,
    fmap_size=25,
    descriptors=[
        WaveKernelSignature.from_registry(n_domain=200),
        HeatKernelSignature.from_registry(n_domain=100),
    ],
    subsamplers=[ArangeSubsampler(subsample_step=8)],
    sdp_weight=1.5,
    lb_weight=5e-3,
    mult_weight=5e-2,
    refiners=[IcpRefiner(nit=8)],
)

# Register it as a preset
MatcherPresets.register("my_custom_preset", custom_config)

print("Custom preset registered!")
print(f"Available presets: {MatcherPresets.list_presets()}")

In [ ]:
# Now you can use your custom preset like any other
config = MatcherPresets.get("my_custom_preset")
matcher = FunctionalMapMatcher(config=config)

result = matcher(mesh_a, mesh_b)
print(f"Result with custom preset: {result.p2p21.shape}")

## Saving and Loading Configurations

You can save configurations to JSON for reproducibility.

In [ ]:
# Save a configuration
config = MatcherPresets.get("standard", sdp_weight=2.0)
# save_config(config, "my_config.json")

print("Configuration saved!")
print("\nNote: Complex objects (descriptors, refiners) are saved as type info only.")
print(
    "For full reproducibility, use presets with overrides or register custom presets."
)

## Best Practices

### When to use each preset:

1. **Development & Debugging**: Start with `minimal` or `quick`
   - Fast iteration
   - Quick sanity checks
   - Parameter exploration

2. **Production Use**: Use `standard`
   - Good balance of speed and accuracy
   - Works well on most datasets
   - Reliable default choice

3. **Benchmarking**: Use `precise`
   - Maximum accuracy
   - Comparable to published results
   - Requires landmarks for best results

4. **Ablation Studies**: Use `no_refinement` and `standard`
   - Compare raw vs. refined results
   - Understand refinement impact

### Customization strategy:

1. Start with the closest preset
2. Override individual parameters as needed
3. If you find a good combination, register it as a custom preset
4. Use grid search (see [23_systematic_experiments.ipynb](./23_systematic_experiments.ipynb)) to optimize from the preset

## Next Steps

- **[23_systematic_experiments.ipynb](./23_systematic_experiments.ipynb)** - Grid search and parameter optimization
- **[19_matcher.ipynb](./19_matcher.ipynb)** - Manual configuration of matchers
- **[21_experiment.ipynb](./21_experiment.ipynb)** - Running experiments